# Модуль 4. Разбор кейсов: Осмысленные дескрипторы

**Модуль 4. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** Те же 12 строк; признаки рассчитываются только для валидного parent.

Посчитать признаки и отличить нулевое значение, пропуск и контекстное предупреждение.

**Workflow:** Стандартизация → признаки с единицами → четыре условия Ro5 → объяснение.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab04_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%204_Descriptor.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab04/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. От структуры к признакам

Дескриптор — функция выбранного графа. Сначала получаем валидный parent, затем рассчитываем массу, липофильность и топологические признаки. В этой таблице признаков остаются только валидные структуры; исходные 12 строк и журнал ошибок сохраняются отдельно. Пропуск не должен становиться химическим нулём.

**Перед запуском:** Сравните этанол и бензол качественно до расчёта.

In [2]:
from chemistry import standardize, descriptors
raw=pd.read_csv(DATA/"molecules.csv",keep_default_na=False)
table=[]
for row in raw.itertuples():
    try:
        parent=standardize(row.smiles).parent_smiles
    except ValueError:
        continue
    table.append({"id":row.id,"parent":parent,**descriptors(parent)})


[13:59:57] Initializing MetalDisconnector
[13:59:57] Running MetalDisconnector
[13:59:57] Initializing Normalizer
[13:59:57] Running Normalizer
[13:59:57] Initializing MetalDisconnector
[13:59:57] Running MetalDisconnector
[13:59:57] Initializing Normalizer
[13:59:57] Running Normalizer
[13:59:57] Running LargestFragmentChooser
[13:59:57] Running Uncharger
[13:59:57] Initializing MetalDisconnector
[13:59:57] Running MetalDisconnector
[13:59:57] Initializing Normalizer
[13:59:57] Running Normalizer
[13:59:57] Initializing MetalDisconnector
[13:59:57] Running MetalDisconnector
[13:59:57] Initializing Normalizer
[13:59:57] Running Normalizer
[13:59:57] Running LargestFragmentChooser
[13:59:57] Running Uncharger
[13:59:57] Initializing MetalDisconnector
[13:59:57] Running MetalDisconnector
[13:59:57] Initializing Normalizer
[13:59:57] Running Normalizer
[13:59:57] Initializing MetalDisconnector
[13:59:57] Running MetalDisconnector
[13:59:57] Initializing Normalizer
[13:59:57] Running Norma

**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. От структуры к признакам — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Таблица с единицами

MolWt имеет единицы g/mol, TPSA — Å²; HBD/HBA являются счётчиками доноров и акцепторов водородной связи. MolLogP — расчётный признак нейтрального представления, не измеренный logD при заданном pH. Сохраняем ID и parent рядом с числом, чтобы можно было проверить, для какой формы оно получено.

**Перед запуском:** Выведите MW, MolLogP, TPSA, HBD, HBA и число предупреждений.

In [3]:
features=pd.DataFrame(table); display(features)
features.to_csv(OUT/"features.csv",index=False)


,id,parent,MolWt,MolLogP,TPSA,HBD,HBA,ro5_violations
0,M01,CCO,46.069,-0.0014,20.23,1,1,0
1,M02,CCO,46.069,-0.0014,20.23,1,1,0
2,M03,CC(=O)O,60.052,0.0909,37.30,1,1,0
3,M04,CC(=O)O,60.052,0.0909,37.30,1,1,0
4,M05,Clc1ccccc1,112.559,2.3400,0.00,0,0,0
5,M06,Oc1ccccc1O,110.112,1.0978,40.46,2,2,0
6,M07,c1ccccc1,78.114,1.6866,0.00,0,0,0
7,M10,C[C@H](O)C(=O)O,90.078,-0.5482,57.53,2,2,0
8,M11,C[C@@H](O)C(=O)O,90.078,-0.5482,57.53,2,2,0
9,M12,CCCC,58.124,1.8064,0.00,0,0,0


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Таблица с единицами — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Пороговые правила и границы

Четыре условия Ro5 проверяются независимо: масса, липофильность, число доноров и акцепторов. Равенство порогу не является нарушением при строгом знаке >. Контрольный полиол нужен, чтобы неполная реализация по одной массе не прошла незаметно. Число нарушений не является вероятностью эффективности или безопасности.

**Перед запуском:** Проверьте полиол, для которого важны HBD/HBA, а не только масса и липофильность.

In [4]:
assert math.isclose(descriptors("CCO")["MolWt"],46.069,abs_tol=0.01)
assert descriptors("c1ccccc1")["TPSA"]==0
polyol=descriptors("OC(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)CO")
assert polyol["HBD"]>5 and polyol["HBA"]>10 and polyol["ro5_violations"]==2
artifact={"valid_rows":len(features),"ethanol":descriptors("CCO"),"polyol":polyol}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Пороговые правила и границы — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

MW этанола 46,069±0,01; TPSA бензола 0; контрольный полиол нарушает HBD/HBA.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':4,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab04/worked/benchmark.json


## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab04_practice.ipynb](lab04_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.